# NBA Playoff Prediction — Data Preprocessing
**Author:** Anthony  
**Project:** NBA Playoff Matchup Predictor  

---
## Files used
| File | What it contains |
|---|---|
| `Regular_Season.csv` | Player-level regular season stats, 2012–2024 |
| `Playoffs.csv` | Player-level playoff stats, 2012–2024 |
| `ranking.csv` | Team standings by date (for seeding) |
| `nba.csv` | Combined regular season + playoff player stats |

> **Fix from group chat:** The original datasets were player-level only — no game matchup results.
> This notebook (a) aggregates players → team-level features and (b) adds a playoff matchup/result table.

In [ ]:
import pandas as pd
import numpy as np
import warnings, os
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

DATA_DIR   = 'data/'    # <-- put CSVs here
OUTPUT_DIR = 'output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

regular_raw = pd.read_csv(f'{DATA_DIR}Regular_Season.csv')
playoff_raw = pd.read_csv(f'{DATA_DIR}Playoffs.csv')
ranking_raw = pd.read_csv(f'{DATA_DIR}ranking.csv')

print('Regular season:', regular_raw.shape)
print('Playoffs:      ', playoff_raw.shape)
print('Ranking:       ', ranking_raw.shape)

---
## Step 1: Clean & Standardize Year
Years come in '2022-23' format. We extract the start year as an integer so everything merges cleanly.

In [ ]:
def parse_year(df):
    df = df.copy()
    # '2022-23' -> 2022
    df['Season'] = df['year'].str[:4].astype(int)
    return df

regular_raw = parse_year(regular_raw)
playoff_raw = parse_year(playoff_raw)

print('Seasons covered:', sorted(regular_raw['Season'].unique()))

---
## Step 2: Aggregate Player Stats → Team Stats

**This is the key fix.** The raw CSVs are player-level. We need team-level averages to predict team vs. team matchups.

In [ ]:
STAT_COLS = ['GP','MIN','FGM','FGA','FG_PCT','FG3M','FG3A','FG3_PCT',
             'FTM','FTA','FT_PCT','OREB','DREB','REB','AST','STL','BLK','TOV','PF','PTS']

def aggregate_to_team(df, label=''):
    """
    Aggregates player-level rows to team-level season averages.
    Uses mean across all players on the roster for each stat.
    Also computes a team_PTS_total = sum of player PTS for context.
    """
    df = df.copy()

    # Mean player stats per team per season
    agg = (
        df.groupby(['Season', 'TEAM_ID', 'TEAM'])[STAT_COLS]
        .mean()
        .reset_index()
    )

    # Rename so we know these are team averages
    rename = {c: f'team_avg_{c}' for c in STAT_COLS}
    agg = agg.rename(columns=rename)

    # Also add total team PTS (sum across players) as a separate feature
    pts_sum = (
        df.groupby(['Season', 'TEAM_ID'])['PTS']
        .sum()
        .reset_index()
        .rename(columns={'PTS': 'team_total_PTS'})
    )
    agg = agg.merge(pts_sum, on=['Season', 'TEAM_ID'], how='left')

    print(f'{label} team stats: {agg.shape} | Seasons: {sorted(agg["Season"].unique())}')
    return agg

team_regular = aggregate_to_team(regular_raw, 'Regular season')
team_playoff = aggregate_to_team(playoff_raw, 'Playoff')
team_regular.head(3)

---
## Step 3: Add End-of-Season Seeding from ranking.csv

In [ ]:
def get_final_standings(ranking_df):
    """
    Gets the final regular season standing (win %) for each team per season.
    SEASON_ID format: 22022 = 2022-23 regular season (starts with '2').
    """
    df = ranking_df.copy()

    # Keep only regular season records (SEASON_ID starts with '2')
    df = df[df['SEASON_ID'].astype(str).str.startswith('2')].copy()

    # Extract year from SEASON_ID: 22022 -> 2022
    df['Season'] = df['SEASON_ID'].astype(str).str[1:].astype(int)

    # Take the latest standing date per team per season (end of regular season)
    df['STANDINGSDATE'] = pd.to_datetime(df['STANDINGSDATE'])
    final = (
        df.sort_values('STANDINGSDATE')
        .groupby(['Season', 'TEAM_ID'])
        .last()
        .reset_index()
    )

    # Rank within conference to get playoff seeding (1 = best)
    final['Conf_Seed'] = (
        final.groupby(['Season', 'CONFERENCE'])['W_PCT']
        .rank(ascending=False, method='first')
        .astype(int)
    )

    return final[['Season', 'TEAM_ID', 'TEAM', 'CONFERENCE', 'W', 'L', 'W_PCT',
                  'HOME_RECORD', 'ROAD_RECORD', 'Conf_Seed']]

standings = get_final_standings(ranking_raw)
print(standings.shape)
standings[standings['Season'] == 2022].sort_values(['CONFERENCE', 'Conf_Seed']).head(10)

---
## Step 4: Build Master Team Stats Table

Merge regular season aggregated stats + standings into one table per team per season.

In [ ]:
master_stats = team_regular.merge(
    standings[['Season','TEAM_ID','CONFERENCE','W','L','W_PCT','Conf_Seed',
               'HOME_RECORD','ROAD_RECORD']],
    on=['Season','TEAM_ID'],
    how='left'
)

# Parse home/road win % from record strings (e.g. '25-16')
def record_to_pct(record_str):
    try:
        w, l = str(record_str).split('-')
        total = int(w) + int(l)
        return int(w) / total if total > 0 else 0.5
    except:
        return np.nan

master_stats['Home_Win_Pct'] = master_stats['HOME_RECORD'].apply(record_to_pct)
master_stats['Road_Win_Pct'] = master_stats['ROAD_RECORD'].apply(record_to_pct)

# Flag which teams made the playoffs
playoff_teams = playoff_raw.groupby(['Season','TEAM_ID'])['PLAYER_ID'].count().reset_index()[['Season','TEAM_ID']]
playoff_teams['Made_Playoffs'] = 1
master_stats = master_stats.merge(playoff_teams, on=['Season','TEAM_ID'], how='left')
master_stats['Made_Playoffs'] = master_stats['Made_Playoffs'].fillna(0).astype(int)

print(f'Master stats: {master_stats.shape}')
print(f'Playoff teams per season: {master_stats.groupby("Season")["Made_Playoffs"].sum().to_dict()}')
master_stats.head(3)

---
## Step 5: Build Playoff Matchup Results Table

**This is the missing piece.** The CSVs don't include who played who in each series or who won.
We hardcode the actual NBA playoff series results for 2012–2024 from historical records.

Each row = one playoff series. `Winner` beat `Loser` in that round that season.

In [ ]:
# NBA Playoff Series Results 2012–2024
# Format: (Season_start_year, Round, Winner_abbr, Loser_abbr)
# Rounds: 1=First Round, 2=Semifinals, 3=Conf Finals, 4=NBA Finals

playoff_series_raw = [
    # 2012-13
    (2012,1,'MIA','MIL'), (2012,1,'CHI','BKN'), (2012,1,'IND','ATL'), (2012,1,'NYK','BOS'),
    (2012,1,'OKC','HOU'), (2012,1,'SAS','LAL'), (2012,1,'GSW','DEN'), (2012,1,'MEM','LAC'),
    (2012,2,'MIA','CHI'), (2012,2,'IND','NYK'), (2012,2,'SAS','GSW'), (2012,2,'MEM','OKC'),
    (2012,3,'MIA','IND'), (2012,3,'SAS','MEM'),
    (2012,4,'MIA','SAS'),
    # 2013-14
    (2013,1,'MIA','CHA'), (2013,1,'BKN','TOR'), (2013,1,'CHI','WAS'), (2013,1,'IND','ATL'),
    (2013,1,'SAS','DAL'), (2013,1,'OKC','MEM'), (2013,1,'LAC','GSW'), (2013,1,'POR','HOU'),
    (2013,2,'MIA','BKN'), (2013,2,'IND','WAS'), (2013,2,'SAS','POR'), (2013,2,'OKC','LAC'),
    (2013,3,'MIA','IND'), (2013,3,'SAS','OKC'),
    (2013,4,'SAS','MIA'),
    # 2014-15
    (2014,1,'CLE','BOS'), (2014,1,'CHI','MIL'), (2014,1,'ATL','BKN'), (2014,1,'WAS','TOR'),
    (2014,1,'GSW','NOP'), (2014,1,'MEM','POR'), (2014,1,'HOU','DAL'), (2014,1,'LAC','SAS'),
    (2014,2,'CLE','CHI'), (2014,2,'ATL','WAS'), (2014,2,'GSW','MEM'), (2014,2,'HOU','LAC'),
    (2014,3,'CLE','ATL'), (2014,3,'GSW','HOU'),
    (2014,4,'GSW','CLE'),
    # 2015-16
    (2015,1,'CLE','DET'), (2015,1,'ATL','BOS'), (2015,1,'MIA','CHA'), (2015,1,'TOR','IND'),
    (2015,1,'GSW','HOU'), (2015,1,'SAS','MEM'), (2015,1,'OKC','DAL'), (2015,1,'POR','LAC'),
    (2015,2,'CLE','ATL'), (2015,2,'TOR','MIA'), (2015,2,'GSW','POR'), (2015,2,'OKC','SAS'),
    (2015,3,'CLE','TOR'), (2015,3,'GSW','OKC'),
    (2015,4,'CLE','GSW'),
    # 2016-17
    (2016,1,'BOS','CHI'), (2016,1,'WAS','ATL'), (2016,1,'TOR','MIL'), (2016,1,'CLE','IND'),
    (2016,1,'GSW','POR'), (2016,1,'SAS','MEM'), (2016,1,'HOU','OKC'), (2016,1,'UTA','LAC'),
    (2016,2,'BOS','WAS'), (2016,2,'CLE','TOR'), (2016,2,'GSW','UTA'), (2016,2,'SAS','HOU'),
    (2016,3,'CLE','BOS'), (2016,3,'GSW','SAS'),
    (2016,4,'GSW','CLE'),
    # 2017-18
    (2017,1,'MIL','BOS'), (2017,1,'PHI','MIA'), (2017,1,'CLE','IND'), (2017,1,'TOR','WAS'),
    (2017,1,'GSW','SAS'), (2017,1,'HOU','MIN'), (2017,1,'POR','NOP'), (2017,1,'OKC','UTA'),
    (2017,2,'BOS','PHI'), (2017,2,'CLE','TOR'), (2017,2,'GSW','NOP'), (2017,2,'HOU','UTA'),
    (2017,3,'CLE','BOS'), (2017,3,'GSW','HOU'),
    (2017,4,'GSW','CLE'),
    # 2018-19
    (2018,1,'MIL','DET'), (2018,1,'BOS','IND'), (2018,1,'PHI','BKN'), (2018,1,'TOR','ORL'),
    (2018,1,'GSW','LAC'), (2018,1,'HOU','UTA'), (2018,1,'POR','OKC'), (2018,1,'DEN','SAS'),
    (2018,2,'MIL','BOS'), (2018,2,'TOR','PHI'), (2018,2,'GSW','HOU'), (2018,2,'POR','DEN'),
    (2018,3,'GSW','POR'), (2018,3,'TOR','MIL'),
    (2018,4,'TOR','GSW'),
    # 2019-20 (bubble)
    (2019,1,'LAL','POR'), (2019,1,'HOU','OKC'), (2019,1,'LAC','DAL'), (2019,1,'DEN','UTA'),
    (2019,1,'MIL','ORL'), (2019,1,'MIA','IND'), (2019,1,'BOS','PHI'), (2019,1,'TOR','BKN'),
    (2019,2,'LAL','HOU'), (2019,2,'DEN','LAC'), (2019,2,'MIA','MIL'), (2019,2,'BOS','TOR'),
    (2019,3,'LAL','DEN'), (2019,3,'MIA','BOS'),
    (2019,4,'LAL','MIA'),
    # 2020-21
    (2020,1,'PHI','WAS'), (2020,1,'MIL','MIA'), (2020,1,'NYK','ATL'), (2020,1,'BKN','BOS'),
    (2020,1,'UTA','MEM'), (2020,1,'DEN','POR'), (2020,1,'LAC','DAL'), (2020,1,'PHX','LAL'),
    (2020,2,'ATL','PHI'), (2020,2,'MIL','BKN'), (2020,2,'PHX','DEN'), (2020,2,'LAC','UTA'),
    (2020,3,'MIL','ATL'), (2020,3,'PHX','LAC'),
    (2020,4,'MIL','PHX'),
    # 2021-22
    (2021,1,'MIA','ATL'), (2021,1,'PHI','TOR'), (2021,1,'MIL','CHI'), (2021,1,'BOS','BKN'),
    (2021,1,'GSW','DEN'), (2021,1,'MEM','MIN'), (2021,1,'PHX','NOP'), (2021,1,'DAL','UTA'),
    (2021,2,'MIA','PHI'), (2021,2,'BOS','MIL'), (2021,2,'GSW','MEM'), (2021,2,'DAL','PHX'),
    (2021,3,'BOS','MIA'), (2021,3,'GSW','DAL'),
    (2021,4,'GSW','BOS'),
    # 2022-23
    (2022,1,'MIA','MIL'), (2022,1,'NYK','CLE'), (2022,1,'BOS','ATL'), (2022,1,'PHI','BKN'),
    (2022,1,'DEN','MIN'), (2022,1,'GSW','SAC'), (2022,1,'LAL','MEM'), (2022,1,'PHX','LAC'),
    (2022,2,'MIA','NYK'), (2022,2,'BOS','PHI'), (2022,2,'DEN','PHX'), (2022,2,'LAL','GSW'),
    (2022,3,'MIA','BOS'), (2022,3,'DEN','LAL'),
    (2022,4,'DEN','MIA'),
    # 2023-24
    (2023,1,'BOS','MIA'), (2023,1,'CLE','ORL'), (2023,1,'NYK','PHI'), (2023,1,'MIL','IND'),
    (2023,1,'OKC','NOP'), (2023,1,'DEN','LAL'), (2023,1,'MIN','PHX'), (2023,1,'DAL','LAC'),
    (2023,2,'BOS','CLE'), (2023,2,'IND','NYK'), (2023,2,'DEN','MIN'), (2023,2,'DAL','OKC'),
    (2023,3,'BOS','IND'), (2023,3,'DAL','MIN'),
    (2023,4,'BOS','DAL'),
]

playoff_matchups = pd.DataFrame(playoff_series_raw, columns=['Season','Round','Winner','Loser'])
playoff_matchups['Round_Name'] = playoff_matchups['Round'].map({
    1:'First Round', 2:'Conf Semifinals', 3:'Conf Finals', 4:'NBA Finals'
})

print(f'Playoff series records: {len(playoff_matchups)}')
print(f'Expected: {len(playoff_series_raw)} ({12*4 + 12}= 60 per 12 years...)')
playoff_matchups.head(10)

---
## Step 6: Map Team IDs onto Matchups

The matchup table uses team abbreviations. We need to join in the TEAM_IDs from master_stats so everything merges cleanly.

In [ ]:
# Build abbreviation → TEAM_ID lookup from master_stats
team_id_lookup = (
    master_stats[['TEAM','TEAM_ID']]
    .drop_duplicates()
    .set_index('TEAM')['TEAM_ID']
    .to_dict()
)

playoff_matchups['Winner_ID'] = playoff_matchups['Winner'].map(team_id_lookup)
playoff_matchups['Loser_ID']  = playoff_matchups['Loser'].map(team_id_lookup)

# Check for any unmapped teams
unmapped = playoff_matchups[playoff_matchups['Winner_ID'].isna() | playoff_matchups['Loser_ID'].isna()]
if len(unmapped) > 0:
    print('Warning: unmapped teams:', unmapped[['Season','Winner','Loser']].values)
else:
    print('All teams mapped successfully.')

playoff_matchups.head(5)

---
## Step 7: Build T1 / T2 Symmetric Training Set

Same dataset-doubling approach as our March ML Mania notebook.
Each series becomes two rows — once from the winner's perspective (Target=1) and once from the loser's (Target=0).

In [ ]:
def build_matchup_dataset(matchups_df, stats_df):
    """
    Merges team stats onto each side of every playoff matchup
    and computes differential features.
    """
    feature_cols = [c for c in stats_df.columns
                    if c not in ['Season','TEAM_ID','TEAM']]

    rows = []
    for _, m in matchups_df.iterrows():
        season = m['Season']
        w_id   = m['Winner_ID']
        l_id   = m['Loser_ID']

        w_stats = stats_df[(stats_df['Season']==season) & (stats_df['TEAM_ID']==w_id)]
        l_stats = stats_df[(stats_df['Season']==season) & (stats_df['TEAM_ID']==l_id)]

        if w_stats.empty or l_stats.empty:
            continue

        w = w_stats.iloc[0]
        l = l_stats.iloc[0]

        # Winner perspective (T1=winner)
        row_w = {'Season': season, 'Round': m['Round'], 'Round_Name': m['Round_Name'],
                 'T1_TeamID': w_id, 'T1_Team': m['Winner'],
                 'T2_TeamID': l_id, 'T2_Team': m['Loser'],
                 'Target_Win': 1}
        for feat in feature_cols:
            row_w[f'T1_{feat}'] = w.get(feat, np.nan)
            row_w[f'T2_{feat}'] = l.get(feat, np.nan)
            row_w[f'{feat}_Diff'] = w.get(feat, np.nan) - l.get(feat, np.nan)

        # Loser perspective (T1=loser)
        row_l = {'Season': season, 'Round': m['Round'], 'Round_Name': m['Round_Name'],
                 'T1_TeamID': l_id, 'T1_Team': m['Loser'],
                 'T2_TeamID': w_id, 'T2_Team': m['Winner'],
                 'Target_Win': 0}
        for feat in feature_cols:
            row_l[f'T1_{feat}'] = l.get(feat, np.nan)
            row_l[f'T2_{feat}'] = w.get(feat, np.nan)
            row_l[f'{feat}_Diff'] = l.get(feat, np.nan) - w.get(feat, np.nan)

        rows.extend([row_w, row_l])

    return pd.DataFrame(rows)

# Clean up master_stats to only the feature columns we want
keep_cols = ['Season','TEAM_ID','TEAM'] + \
            [c for c in master_stats.columns if c.startswith('team_avg_')] + \
            ['team_total_PTS','W_PCT','Home_Win_Pct','Road_Win_Pct','Conf_Seed','Made_Playoffs']
master_clean = master_stats[[c for c in keep_cols if c in master_stats.columns]].copy()

playoff_dataset = build_matchup_dataset(playoff_matchups, master_clean)

print(f'Playoff matchup dataset: {playoff_dataset.shape}')
print(f'Win rate (should be 0.5): {playoff_dataset["Target_Win"].mean()}')
playoff_dataset.head(4)

---
## Step 8: Build Feature & Target Arrays

In [ ]:
# Use differential features as model inputs
diff_features = [c for c in playoff_dataset.columns if c.endswith('_Diff')]
print(f'{len(diff_features)} differential features:')
print(diff_features)

X = playoff_dataset[diff_features].fillna(0)
y = playoff_dataset['Target_Win']
meta = playoff_dataset[['Season','Round','Round_Name','T1_Team','T2_Team']]

print(f'\nX shape: {X.shape}')
print(f'Missing values: {X.isnull().sum().sum()}')

---
## Step 9: Save Outputs

In [ ]:
master_clean.to_csv(f'{OUTPUT_DIR}master_stats.csv', index=False)
playoff_matchups.to_csv(f'{OUTPUT_DIR}playoff_matchups.csv', index=False)
playoff_dataset.to_csv(f'{OUTPUT_DIR}playoff_dataset.csv', index=False)
X.to_csv(f'{OUTPUT_DIR}X_train.csv', index=False)
y.to_csv(f'{OUTPUT_DIR}y_train.csv', index=False)
meta.to_csv(f'{OUTPUT_DIR}meta.csv', index=False)

print('Saved to output/:')
print(f'  master_stats.csv      {master_clean.shape}')
print(f'  playoff_matchups.csv  {playoff_matchups.shape}')
print(f'  playoff_dataset.csv   {playoff_dataset.shape}')
print(f'  X_train.csv           {X.shape}')
print(f'  y_train.csv           {y.shape}')

---
## Notes for Olivia & Benjamin

Files ready for `nba_modeling.ipynb`:

- `X_train.csv` — differential features for each playoff series matchup (both perspectives)
- `y_train.csv` — 1 if T1 won, 0 if T1 lost
- `meta.csv` — season, round, team names for each row (for analysis/display)
- `master_stats.csv` — full per-team per-season feature table
- `playoff_matchups.csv` — raw series results with team IDs

**Key difference from old notebook:** Training data is now **playoff series** (not regular season games), so each row = one series matchup. The model learns which team traits predict winning a playoff series — which is exactly what we want.